# 03 · Content Moderation & Toxicity
### *Ethics, Safety & the Future of LLMs — Unit 3*

Deployed systems wrap the model in a **moderation pipeline** — filtering what goes in and what comes out. Here we
build the core pieces:

1. Score text with a real **toxicity classifier**.
2. Build a **moderation gate** (allow / block).
3. Detect and **redact PII** with regex.

> CPU is fine. Nothing offensive is generated — we only *score* example strings.

In [ ]:
!pip -q install "transformers>=4.40"

## 1 · A real toxicity classifier

`unitary/toxic-bert` returns scores for several toxicity categories. We score a few neutral and mildly rude
example strings (kept tame on purpose).

In [ ]:
from transformers import pipeline
tox = pipeline("text-classification", model="unitary/toxic-bert", top_k=None)

samples = [
    "Thanks so much, this was really helpful!",
    "You are being unreasonable and I disagree strongly.",
    "I can't stand this stupid broken product.",
    "Have a wonderful day everyone.",
]
for s in samples:
    scores = {d["label"]: round(d["score"],3) for d in tox(s)[0]}
    top = max(scores, key=scores.get)
    print(f"[{top:12s} {scores[top]:.2f}]  {s}")

## 2 · A moderation gate

A simple policy: if the top toxicity score crosses a threshold, **block** the message; otherwise allow it.
Real systems use per-category thresholds and human review for edge cases.

In [ ]:
def moderate(text, threshold=0.5):
    scores = {d["label"]: d["score"] for d in tox(text)[0]}
    worst_label = max(scores, key=scores.get)
    worst = scores[worst_label]
    decision = "BLOCK" if worst >= threshold else "ALLOW"
    return decision, worst_label, round(worst,2)

for s in samples:
    print(moderate(s), "|", s)

## 3 · PII detection & redaction

Moderation isn't only toxicity — leaking personal data is a safety failure too. A regex layer catches common PII
before it is logged or returned.

In [ ]:
import re
PII = {
    "EMAIL":       r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}",
    "PHONE":       r"\b(?:\+?\d{1,3}[- ]?)?(?:\(?\d{3}\)?[- ]?)\d{3}[- ]?\d{4}\b",
    "CREDIT_CARD": r"\b(?:\d[ -]?){13,16}\b",
    "SSN":         r"\b\d{3}-\d{2}-\d{4}\b",
}
def redact(text):
    found = {}
    for name, pat in PII.items():
        for m in re.findall(pat, text):
            found.setdefault(name, []).append(m)
            text = text.replace(m if isinstance(m,str) else m[0], f"[{name}_REDACTED]")
    return text, found

msg = "Email me at jane.doe@example.com or call 415-555-0198. My SSN is 123-45-6789."
clean, found = redact(msg)
print("REDACTED:", clean)
print("FOUND   :", found)

## Recap & your turn

- A moderation pipeline is **input filter → model → output filter**, not just a model.
- Automatic classifiers are useful but imperfect — tune thresholds and keep human review.
- **PII redaction** protects privacy at the boundary of the system.

**Exercises**
1. Add per-category thresholds (e.g. stricter for `threat` than `insult`).
2. Combine both: moderate a message, and if allowed, still redact PII before storing it.
3. Explore `allenai/real-toxicity-prompts` to see how base models can be steered toward toxic completions.